In [1]:
import pandas as pd

In [2]:
X = pd.read_csv("../data/secom.data", sep=r"\s+", header=None)
X.columns = [f"sensor_{i}" for i in range(X.shape[1])]

labels = pd.read_csv("../data/secom_labels.data", sep=r"\s+", header=None,
                     names=["result", "timestamp_raw"])
labels["timestamp"] = pd.to_datetime(labels["timestamp_raw"], format="%d/%m/%Y %H:%M:%S")
labels["fail"] = labels["result"] == 1  # label is -1 for pass, 1 for fail

In [3]:
print(X.shape)
print(labels["fail"].mean())
print(labels["timestamp"].min(), labels["timestamp"].max())
print(X.isna().mean().describe())  # fraction missing per sensor

(1567, 590)
0.06636885768985322
2008-07-19 11:55:00 2008-10-17 06:07:00
count    590.000000
mean       0.045375
std        0.154340
min        0.000000
25%        0.001276
50%        0.003829
75%        0.005743
max        0.911934
dtype: float64


## Initial data exploration

- Size: 1,567 production runs, each with 590 sensor measurements
- Fail rate: 6.6% (about 104 failed runs), so failures are rare relative to passes
- Time span: 19 July 2008 to 17 October 2008 (about three months of production)
- Missing data: most sensors are nearly complete (median 0.4% missing), but a small number are heavily missing (up to 91%)

In [4]:
missing = X.isna().mean()
# drop sensors that are more than half missing, then ones that never change
X1 = X.loc[:, missing <= 0.5]
constant = X1.nunique(dropna=True) <= 1
X2 = X1.loc[:, ~constant]

print(f"{X.shape[1]} sensors -> {X1.shape[1]} after missing filter -> {X2.shape[1]} after constant filter")

590 sensors -> 562 after missing filter -> 446 after constant filter


In [5]:
df = X2.copy()
df["timestamp"] = labels["timestamp"]
df["fail"] = labels["fail"]
df = df.sort_values("timestamp")
df.to_csv("../data/secom_clean.csv", index=False)

## Data cleaning

- Removed 28 sensors with more than 50% missing values, as too little data remained to monitor them reliably.
- Removed 116 sensors with a constant value across all runs, as they carry no information about process variation.
- 446 sensors remain (from 590). Remaining missing values (generally under 1% per sensor) are left in place and handled sensor by sensor in later analysis.